# Retail Loan Portfolio Stress Testing
Analyzing potential portfolio losses under baseline, adverse, and extreme economic scenarios using predicted probabilities of default.

#### Predicted Probability of Default (PD)
In a previous project, I built a logistic regression model to predict the likelihood that a borrower may default. The predicted PD values for each borrower were saved in a CSV file (predicted_PD.csv), which we now use as the starting point for stress testing.

Each row in the dataset represents a borrower, and the prob_default column shows the likelihood of default:

In [67]:
import pandas as pd

# Load PDs
predicted_PD = pd.read_csv("predicted_PD.csv")

# Check the first few rows
print(predicted_PD.head())

   prob_default
0      0.146211
1      0.466748
2      0.241518
3      0.100798
4      0.090225


In this step, we load the predicted probabilities of default (PD) that were generated from our logistic regression model.

#### Add predicted PD to the original dataset

In [73]:
# Load original dataset 
original_data = pd.read_csv(r"C:\Users\BINARY\Downloads\cr_loan2.csv")

# Add the predicted PDs to the original loan data
original_data['prob_default'] = predicted_PD['prob_default']

#### Define stress scenarios

In [69]:
# Define stress scenarios
scenarios = {
    "baseline": {"income_decrease": 0, "interest_increase": 0},
    "adverse":  {"income_decrease": 0.10, "interest_increase": 0.02},
    "extreme":  {"income_decrease": 0.20, "interest_increase": 0.04}
}

I simulate three scenarios to see how the portfolio behaves under different economic conditions:

Baseline: no change in income or interest rates (0% income decrease, 0% interest increase)

Adverse: moderate stress (10% income decrease, 2% interest increase)

Extreme: severe stress (20% income decrease, 4% interest increase)

These changes are applied to the predicted default probabilities to calculate stressed expected losses.

### Apply stress to PD

In [75]:
# Calculate stressed PDs
for scenario, shock in scenarios.items():
    original_data[f'PD_{scenario}'] = original_data['prob_default'] * (
        1 + shock["income_decrease"] + shock["interest_increase"]
    )
  
    original_data[f'PD_{scenario}'] = original_data[f'PD_{scenario}'].clip(upper=1)

We increase each borrower's PD according to the scenario. This simulates how risk rises if conditions worsen.

### Calculate Expected Loss

In [71]:
LGD = 0.40  # 40% loss given default

for scenario in scenarios.keys():
    original_data[f'EL_{scenario}'] = original_data[f'PD_{scenario}'] * LGD * original_data['loan_amnt']

The Expected Loss (EL) represents the potential loss a lender may face if a borrower defaults. It is calculated using the formula:

##### EL = PD × LGD × EAD

Where:

PD (Probability of Default) – the likelihood that a borrower will default, adjusted for stress scenarios (baseline, adverse, extreme).

LGD (Loss Given Default) – the proportion of the loan that would be lost if default occurs. Since we do not have detailed collateral or exposure information, we assume LGD = 40% (0.40) as a common conservative estimate.

EAD (Exposure at Default) – the total loan amount for each borrower.

### Aggregate Portfolio Loss

In [72]:
print("Total Portfolio Expected Loss:")
for scenario in scenarios.keys():
    print(f"{scenario.capitalize()}:", original_data[f'EL_{scenario}'].sum())

Total Portfolio Expected Loss:
Baseline: 9114400.634858439
Adverse: 10206604.394289223
Extreme: 11277484.910408348


## Conclusion:
The stress testing shows that the bank’s potential losses increase significantly under worsening economic conditions. While the baseline scenario results in a total expected loss of around 9.1M, the adverse scenario rises to 10.2M, and the extreme scenario reaches 11.2M. This highlights the importance of monitoring credit risk and preparing for economic downturns to ensure financial stability.